# Demo 01 : Graphs and state machines

Run the cells in order and watch. There is nothing to write.

## a · directed-graphs

We represent each step as a node and each allowed transition as a directed edge.

In [1]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

class Job(TypedDict):
    status: str
    history: list[str]

def step(status: str):
    def update(state: Job) -> dict:
        return {"status": status, "history": [*state["history"], status]}
    return update

graph = StateGraph(Job)
graph.add_node("diagnose", step("diagnosed"))
graph.add_node("parts", step("waiting for parts"))
graph.add_node("repair", step("closed"))
graph.add_edge(START, "diagnose")
graph.add_edge("diagnose", "parts")
graph.add_edge("parts", "repair")
graph.add_edge("repair", END)
lifecycle = graph.compile()
print("history:", lifecycle.invoke({"status": "open", "history": ["open"]})["history"])
print(lifecycle.get_graph().draw_mermaid())

history: ['open', 'diagnosed', 'waiting for parts', 'closed']
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	diagnose(diagnose)
	parts(parts)
	repair(repair)
	__end__([<p>__end__</p>]):::last
	__start__ --> diagnose;
	diagnose --> parts;
	parts --> repair;
	repair --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## b · work-order-lifecycle

We add a retry path back to diagnosis. The first diagnosis fails, so the
back edge runs once. The router leaves the loop once `attempts` reaches 2 :
a cap in state, not a hope that the model succeeds.

In [2]:
from typing import Literal

class WorkOrder(TypedDict):
    status: str
    attempts: int
    history: list[str]

def diagnose(state: WorkOrder) -> dict:
    attempts = state["attempts"] + 1
    status = "diagnosed" if attempts >= 2 else "diagnosis failed"
    return {"status": status, "attempts": attempts, "history": [*state["history"], status]}

def route(state: WorkOrder) -> Literal["parts", "diagnose"]:
    return "parts" if state["attempts"] >= 2 else "diagnose"

loop = StateGraph(WorkOrder)
loop.add_node("diagnose", diagnose)
loop.add_node("parts", lambda s: {"status": "waiting for parts", "history": [*s["history"], "waiting for parts"]})
loop.add_edge(START, "diagnose")
loop.add_conditional_edges("diagnose", route)
loop.add_edge("parts", END)
result = loop.compile().invoke({"status": "open", "attempts": 0, "history": ["open"]})
print("history:", result["history"])
print("status:", result["status"], "attempts:", result["attempts"])

history: ['open', 'diagnosis failed', 'diagnosed', 'waiting for parts']
status: waiting for parts attempts: 2


## Things to try

- Add an inspection node between diagnosis and parts
- Route a second failed diagnosis to a manual review node